# 13 · VLM 接口：从场景描述到结构化驾驶条件

VLM 在智驾系统中更适合作为高层语义和结构化条件提供者，而不是直接输出油门和方向盘。本 notebook 不调用外部模型，而是模拟一个带置信度和 abstain 的 VLM 输出，练习 schema、评测和 planner gate。

学习目标：

- 设计稳定的 structured scene condition 接口；
- 区分语义识别、置信度、拒答和控制权限；
- 计算不同 confidence threshold 下的 precision、recall 和 coverage；
- 明确 VLM 误判如何被安全层隔离。


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

rng = np.random.default_rng(51)
LABELS = ['red_light', 'construction', 'occluded_pedestrian']

def make_scene_labels(n=800, seed=51):
    local = np.random.default_rng(seed)
    truth = np.c_[
        local.random(n) < 0.22,
        local.random(n) < 0.16,
        local.random(n) < 0.12,
    ]
    return truth

def mock_vlm(truth, noise=0.12, abstain_rate=0.10, seed=51):
    local = np.random.default_rng(seed)
    probability = truth.astype(float) * (1 - noise) + (~truth).astype(float) * noise
    probability += local.normal(0, 0.08, size=probability.shape)
    probability = np.clip(probability, 0, 1)
    abstain = local.random(truth.shape) < abstain_rate
    return probability, abstain

truth = make_scene_labels()
probability, abstain = mock_vlm(truth)
print('truth shape:', truth.shape)
print('example structured record:', json.dumps({
    'scene_id': 0,
    'conditions': {label: bool(probability[0, i] >= 0.5) for i, label in enumerate(LABELS)},
    'confidence': {label: round(float(probability[0, i]), 3) for i, label in enumerate(LABELS)},
    'abstain': {label: bool(abstain[0, i]) for i, label in enumerate(LABELS)},
}, ensure_ascii=False))


In [ ]:
def validate_record(record):
    required = {'scene_id', 'conditions', 'confidence', 'abstain'}
    if set(record) != required:
        return False, 'top-level keys mismatch'
    if set(record['conditions']) != set(LABELS):
        return False, 'condition keys mismatch'
    if set(record['confidence']) != set(LABELS):
        return False, 'confidence keys mismatch'
    if set(record['abstain']) != set(LABELS):
        return False, 'abstain keys mismatch'
    if not all(isinstance(value, bool) for value in record['conditions'].values()):
        return False, 'conditions must be bool'
    if not all(0 <= value <= 1 for value in record['confidence'].values()):
        return False, 'confidence must be in [0,1]'
    return True, 'ok'

records = []
for scene_id in range(len(truth)):
    record = {
        'scene_id': scene_id,
        'conditions': {label: bool(probability[scene_id, i] >= 0.5) for i, label in enumerate(LABELS)},
        'confidence': {label: float(probability[scene_id, i]) for i, label in enumerate(LABELS)},
        'abstain': {label: bool(abstain[scene_id, i]) for i, label in enumerate(LABELS)},
    }
    records.append(record)
print('valid records:', sum(validate_record(record)[0] for record in records), '/', len(records))


In [ ]:
def evaluate_structured(truth, probability, abstain, threshold=0.5):
    prediction = (probability >= threshold) & ~abstain
    rows = []
    for i, label in enumerate(LABELS):
        tp = np.sum(prediction[:, i] & truth[:, i])
        fp = np.sum(prediction[:, i] & ~truth[:, i])
        fn = np.sum(~prediction[:, i] & truth[:, i])
        coverage = np.mean(~abstain[:, i])
        rows.append({
            'label': label,
            'precision': tp / max(tp + fp, 1),
            'recall': tp / max(tp + fn, 1),
            'coverage': coverage,
        })
    return pd.DataFrame(rows)

print(evaluate_structured(truth, probability, abstain).round(3).to_string(index=False))


In [ ]:
def show_vlm(noise=0.12, abstain_rate=0.10, threshold=0.5):
    probability, abstain = mock_vlm(truth, noise=noise, abstain_rate=abstain_rate)
    report = evaluate_structured(truth, probability, abstain, threshold)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    report.set_index('label')[['precision', 'recall', 'coverage']].plot(kind='bar', ax=ax[0])
    ax[0].set_ylim(0, 1.05)
    ax[0].set_title('structured VLM interface')
    ax[0].tick_params(axis='x', rotation=30)
    for i, label in enumerate(LABELS):
        ax[1].hist(probability[truth[:, i], i], bins=15, alpha=0.45, label=f'{label}: positive')
        ax[1].hist(probability[~truth[:, i], i], bins=15, alpha=0.25, label=f'{label}: negative')
    ax[1].axvline(threshold, color='black', linestyle='--')
    ax[1].set_title('confidence distributions')
    ax[1].set_xlabel('confidence')
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(report.round(3).to_string(index=False))

interact(
    show_vlm,
    noise=FloatSlider(min=0.0, max=0.45, step=0.03, value=0.12, description='VLM noise'),
    abstain_rate=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.10, description='abstain'),
    threshold=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5, description='threshold'),
);


### 练习：VLM 输出不能直接绕过安全层

- 增大 noise，找到一个高 coverage 但危险 recall 下降的 operating point。
- 提高 threshold 或 abstain_rate，观察 coverage–precision–recall 的 trade-off。
- 增加一个 reason 字段和 evidence span，要求 planner 只接受 schema-valid 且 confidence 足够的条件。
- 设计一个硬规则：red_light 条件只能影响行为约束，不能直接产生 control command。


In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
coverage, macro_recall = [], []
for threshold in thresholds:
    report = evaluate_structured(truth, probability, abstain, threshold)
    coverage.append(report['coverage'].mean())
    macro_recall.append(report['recall'].mean())
plt.plot(coverage, macro_recall, marker='o')
plt.xlabel('mean coverage')
plt.ylabel('macro recall')
plt.title('VLM structured interface: coverage vs recall')
plt.show()


## 完成标准

- 定义一个 schema-valid 的 structured condition record。
- 报告每个语义条件的 precision、recall、coverage。
- 展示一个 VLM 误判被 planner/safety layer 拦截的例子。
- 下一步用 Hugging Face VLM 替换 mock_vlm，并单独评估 grounding、校准和 abstention；不要把语言流畅度当成驾驶安全指标。
